In [1]:
# Welcome to your new notebook
# Type here in the cell editor to add code!
from pyspark.sql.functions import *
from pyspark.sql.types import *

StatementMeta(, 843f6343-dd36-4f8f-be4e-9aaf907f842b, 3, Finished, Available, Finished, False)

In [2]:
silver_df = spark.table("silver_netflix_titles")

#display(silver_df)
silver_df = silver_df.filter(~col("show_id").isin("s5795", "s5542","s5814"))
#display(silver_df.orderBy(col("rating")))

StatementMeta(, 843f6343-dd36-4f8f-be4e-9aaf907f842b, 4, Finished, Available, Finished, False)

In [3]:
dim_type = (
    silver_df
    .select("type")
    .distinct()
    .orderBy("type")
)

StatementMeta(, 843f6343-dd36-4f8f-be4e-9aaf907f842b, 5, Finished, Available, Finished, False)

In [4]:
from pyspark.sql.window import Window

window = Window.orderBy("type")

dim_type = (
    dim_type
    .withColumn(
        "type_key",
        row_number().over(window)
    )
)

StatementMeta(, 843f6343-dd36-4f8f-be4e-9aaf907f842b, 6, Finished, Available, Finished, False)

In [5]:
(
    dim_type.write
    .mode("overwrite")
    .saveAsTable("dim_type")
)

#spark.sql("SELECT * FROM dim_type").show()

StatementMeta(, 843f6343-dd36-4f8f-be4e-9aaf907f842b, 7, Finished, Available, Finished, False)

In [6]:
window = Window.orderBy("rating")

dim_rating = (
    silver_df
    .select("rating")
    .distinct()
    .withColumn(
        "rating_key",
        row_number().over(window)
    )
)

dim_rating.write.mode("overwrite").saveAsTable("dim_rating")
#spark.sql("SELECT * FROM dim_rating where rating in ('84 min','77 min','66 min')").show()

StatementMeta(, 843f6343-dd36-4f8f-be4e-9aaf907f842b, 8, Finished, Available, Finished, False)

In [7]:
dim_country = (
    silver_df
    .select(
        explode("country_array").alias("country")
    )
    .distinct()
)

window = Window.orderBy("country")

dim_country = (
    dim_country
    .withColumn(
        "country_key",
        row_number().over(window)
    )
)

dim_country.write.mode("overwrite").saveAsTable("dim_country")
#spark.sql("SELECT distinct country FROM dim_country").show()

StatementMeta(, 843f6343-dd36-4f8f-be4e-9aaf907f842b, 9, Finished, Available, Finished, False)

In [8]:
dim_genre = (
    silver_df
    .select(
        explode("genre_array").alias("genre")
    )
    .distinct()
)

window = Window.orderBy("genre")

dim_genre = (
    dim_genre
    .withColumn(
        "genre_key",
        row_number().over(window)
    )
)

dim_genre.write.mode("overwrite").saveAsTable("dim_genre")
#spark.sql("SELECT distinct genre FROM dim_genre").show()

StatementMeta(, 843f6343-dd36-4f8f-be4e-9aaf907f842b, 10, Finished, Available, Finished, False)

In [9]:
dim_date = (
    silver_df
    .select(
        "date_added",
        "added_year",
        "added_month",
        "added_month_name"
    )
    .dropDuplicates(["date_added"])
)

window = Window.orderBy("date_added")

dim_date = (
    dim_date
    .withColumn(
        "date_key",
        row_number().over(window)
    )
)
spark.sql("DROP TABLE IF EXISTS dim_date")
dim_date.write.mode("overwrite").saveAsTable("dim_date")

StatementMeta(, 843f6343-dd36-4f8f-be4e-9aaf907f842b, 11, Finished, Available, Finished, False)

In [10]:
fact_df=silver_df

fact_df = silver_df.join(
    dim_type.select("type", "type_key"),
    on="type",
    how="left"
)

fact_df = fact_df.join(
    dim_rating.select("rating", "rating_key"),
    on="rating",
    how="left"
)

#display(dim_date)

fact_df = fact_df.join(
    dim_date.select("date_added", "date_key"),
    on="date_added",
    how="left"
)
#display(fact_df)

StatementMeta(, 843f6343-dd36-4f8f-be4e-9aaf907f842b, 12, Finished, Available, Finished, False)

In [11]:
fact_titles = fact_df.select(
    "show_id",
    "title",
    "release_year",
    "duration",
    "description",
    "content_age",
    "type_key",
    "rating_key",
    "date_key",
    "genre_array",
    "country_array",
    "ingestion_timestamp",
    "silver_processed_time"
)
fact_titles.write.mode("overwrite").saveAsTable("fact_titles")

StatementMeta(, 843f6343-dd36-4f8f-be4e-9aaf907f842b, 13, Finished, Available, Finished, False)

In [12]:
bridge_title_genre = (
    silver_df
    .select(
        "show_id",
        explode("genre_array").alias("genre")
    )
    .join(dim_genre, "genre", "left")
    .select("show_id", "genre_key")
)

bridge_title_genre.write.mode("overwrite").saveAsTable("bridge_title_genre")

StatementMeta(, 843f6343-dd36-4f8f-be4e-9aaf907f842b, 14, Finished, Available, Finished, False)

In [13]:
bridge_title_country = (
    silver_df
    .select(
        "show_id",
        explode("country_array").alias("country")
    )
    .join(dim_country, "country", "left")
    .select("show_id", "country_key")
)

bridge_title_country.write.mode("overwrite").saveAsTable("bridge_title_country")

StatementMeta(, 843f6343-dd36-4f8f-be4e-9aaf907f842b, 15, Finished, Available, Finished, False)

In [16]:
%%sql
OPTIMIZE fact_titles;
OPTIMIZE dim_country;
OPTIMIZE dim_genre;
OPTIMIZE dim_rating;
OPTIMIZE dim_type;
OPTIMIZE dim_date;

StatementMeta(, 843f6343-dd36-4f8f-be4e-9aaf907f842b, 23, Finished, Available, Finished, True)

<Spark SQL result set with 1 rows and 2 fields>

<Spark SQL result set with 1 rows and 2 fields>

<Spark SQL result set with 1 rows and 2 fields>

<Spark SQL result set with 1 rows and 2 fields>

<Spark SQL result set with 1 rows and 2 fields>

<Spark SQL result set with 1 rows and 2 fields>

In [17]:
%%sql
CREATE OR REPLACE VIEW vw_movies_by_year AS
SELECT
    release_year,
    COUNT(*) AS total_titles
FROM fact_titles
GROUP BY release_year
ORDER BY release_year;

CREATE OR REPLACE VIEW vw_country_summary AS
SELECT
    dc.country,
    COUNT(*) AS total_titles
FROM bridge_title_country btc
INNER JOIN dim_country dc
    ON btc.country_key = dc.country_key
GROUP BY dc.country
ORDER BY total_titles DESC;

CREATE OR REPLACE VIEW vw_genre_summary AS
SELECT
    dg.genre,
    COUNT(*) AS total_titles
FROM bridge_title_genre btg
INNER JOIN dim_genre dg
    ON btg.genre_key = dg.genre_key
GROUP BY dg.genre
ORDER BY total_titles DESC;

CREATE OR REPLACE VIEW vw_rating_summary AS
SELECT
    dr.rating,
    COUNT(*) AS total_titles
FROM fact_titles ft
INNER JOIN dim_rating dr
    ON ft.rating_key = dr.rating_key
GROUP BY dr.rating
ORDER BY total_titles DESC;

CREATE OR REPLACE VIEW vw_monthly_additions AS
SELECT
    dd.added_year,
    dd.added_month,
    dd.added_month_name,
    COUNT(*) AS total_titles
FROM fact_titles ft
INNER JOIN dim_date dd
    ON ft.date_key = dd.date_key
GROUP BY
    dd.added_year,
    dd.added_month,
    dd.added_month_name
ORDER BY
    dd.added_year,
    dd.added_month;

StatementMeta(, 843f6343-dd36-4f8f-be4e-9aaf907f842b, 28, Finished, Available, Finished, True)

<Spark SQL result set with 0 rows and 0 fields>

<Spark SQL result set with 0 rows and 0 fields>

<Spark SQL result set with 0 rows and 0 fields>

<Spark SQL result set with 0 rows and 0 fields>

<Spark SQL result set with 0 rows and 0 fields>